# RL Experiment 07: Full Ablation Study

**Self-contained experiment notebook using DRY architecture.**

## Architecture (DRY Principle)

| Location | What | Example |
|----------|------|---------|
| `schedule_engine/notebooks/` | Reusable functions | `run_ablation()`, `load_context()` |
| `src/schedule_engine/rl/` | Production RL components | PPO, DQN, Random agents |
| **This notebook** | Experiment-specific config | Method comparison setup |

## Experiment Overview
- **Goal**: Systematic comparison across RL methods
- **Methods**: Random (baseline), PPO, DQN
- **Trials**: Multiple runs per method for statistical significance
- **Metrics**: Best fitness, convergence generation across trials

## 1. Imports (from `schedule_engine/notebooks/`)

In [ ]:
from __future__ import annotations
from datetime import datetime
from pathlib import Path

# DRY IMPORTS FROM schedule_engine/notebooks/
from schedule_engine.notebooks import run_ablation

print(" All imports from schedule_engine/notebooks/ successful!")

## 2. Configuration (Inline - Experiment-Specific)

In [ ]:
# ============================================================================
# RL EXPERIMENT 07 CONFIGURATION - Full Ablation Study
# ============================================================================

SEED = 42
POP_SIZE = 20
MAX_GENERATIONS = 50
MAX_STEPS = 20
TIMESTEPS = 3000  # Per method
TRIALS = 5  # Statistical significance

# Methods to compare
METHODS = {
    "random": {"agent_type": "random"},
    "ppo": {"agent_type": "ppo"},
    "dqn": {"agent_type": "dqn"},
}

# Paths - Organized by experiment with timestamp
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
DATA_DIR = Path("../data")
OUTPUT_DIR = Path(f"../output/notebooks/rl_07_ablation_{TIMESTAMP}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f" Config: methods={list(METHODS.keys())}, trials={TRIALS}")
print(f" Output: {OUTPUT_DIR}")

## 3. Run Ablation Study

In [ ]:
# Run systematic ablation across all methods
print(f"Running ablation study: {len(METHODS)} methods × {TRIALS} trials...")
print(f"This may take several minutes...\n")

results = run_ablation(
    methods=METHODS,
    data_dir=DATA_DIR,
    trials=TRIALS,
    timesteps=TIMESTEPS,
    pop_size=POP_SIZE,
    max_generations=MAX_GENERATIONS,
    max_steps=MAX_STEPS,
    seed=SEED,
)

print(f"\n Ablation study completed")

## 4. Results Summary

In [ ]:
import numpy as np

print(f"\n{'='*70}")
print(f"RL EXPERIMENT 07: FULL ABLATION STUDY RESULTS")
print(f"{'='*70}")

# Compute statistics per method
method_stats = {}
for method_key, runs in results.items():
    best_fitness_vals = [r.best_fitness for r in runs]
    conv_vals = [r.convergence_gen for r in runs]
    
    method_stats[method_key] = {
        "best_fitness_mean": np.mean(best_fitness_vals),
        "best_fitness_std": np.std(best_fitness_vals),
        "convergence_mean": np.mean(conv_vals),
        "convergence_std": np.std(conv_vals),
        "best_fitness_all": best_fitness_vals,
        "convergence_all": conv_vals,
    }
    
    print(f"\n{method_key.upper()}:")
    print(f"  Best Fitness: {np.mean(best_fitness_vals):.2f} ± {np.std(best_fitness_vals):.2f}")
    print(f"  Convergence:  {np.mean(conv_vals):.1f} ± {np.std(conv_vals):.1f} generations")
    print(f"  Raw values:   fitness={best_fitness_vals}, conv={conv_vals}")

print(f"\n{'='*70}")

## 5. Save Results

In [ ]:
import json

# Save experiment results
results_data = {
    "experiment": "rl_07_full_ablation_study",
    "timestamp": TIMESTAMP,
    "config": {
        "seed": SEED,
        "pop_size": POP_SIZE,
        "max_generations": MAX_GENERATIONS,
        "max_steps": MAX_STEPS,
        "timesteps": TIMESTEPS,
        "trials": TRIALS,
        "methods": list(METHODS.keys()),
    },
    "results": {
        method: {
            "best_fitness_mean": stats["best_fitness_mean"],
            "best_fitness_std": stats["best_fitness_std"],
            "convergence_mean": stats["convergence_mean"],
            "convergence_std": stats["convergence_std"],
            "best_fitness_all": stats["best_fitness_all"],
            "convergence_all": stats["convergence_all"],
        }
        for method, stats in method_stats.items()
    },
}

results_path = OUTPUT_DIR / "results.json"
with open(results_path, "w") as f:
    json.dump(results_data, f, indent=2)

print(f" Results saved to: {results_path}")